# Symbolic AI Concepts (Search Mini-Notebook)
## Heuristic Search: BFS, DFS, A\*, **Beam Search** — and why it still matters in the LLM era

Modern LLM systems often *look* like they “reason”, but many strong systems rely on **search**:
- **Planning**: explore action sequences to reach a goal (tool-use, web actions, API calls).
- **Verification**: search for counterexamples, constraint violations, or alternative proofs.
- **Reliability**: when a single-shot answer is brittle, search provides **backtracking** and **best-first exploration**.

In this notebook you will:
- implement **BFS**, **DFS**, and **A\*** (minimal, readable Python),
- add **Beam Search** as an LLM-relevant bounded search,
- run them on a small **grid world**,
- compare **path quality** and **states expanded**,
- see how a **heuristic / scoring function** changes the game,
- and get practical library recommendations.

> **No external dependencies** required (pure Python).


## Agenda
1. Search framing: states, edges, costs, goals
2. Problem setup: a grid world with obstacles
3. BFS (optimal for unweighted graphs)
4. DFS (fast to find *a* solution, not necessarily good)
5. A\* (best-first with heuristics; optimal with admissible heuristic)
6. Heuristics as “guidance” (LLM analogy)
7. **Beam search** (bounded best-first, strongly related to LLM decoding / ToT)
8. Why search matters for LLM agents
9. Practical solver/library recommendations


---
## 0) Search as a universal abstraction

We’ll model problems as:
- **State**: what the world looks like right now (e.g., agent position, tool outputs, memory)
- **Actions / transitions**: how you can move to a new state
- **Goal test**: how you know you’re done
- **Cost**: optionally, how expensive actions are

Algorithms differ mainly in *which frontier node they expand next*:
- **BFS**: increasing depth (shortest #steps in unweighted graphs)
- **DFS**: deep first (low memory, can be unlucky)
- **A\***: best-first by `f(n)=g(n)+h(n)`  
- **Beam search**: like best-first, but keeps only the **top B** partial candidates at each step (bounded memory).


---
## 1) Grid world problem

We’ll plan from **Start** to **Goal** in a 2D grid with obstacles.
Moves are 4-connected (up/down/left/right), each with cost 1.

This toy example maps to LLM agents:
- a “state” could be the **current plan prefix** or **current environment snapshot**
- an “action” could be **calling a tool**, **clicking a UI element**, **asking a sub-question**
- the goal could be a **verified final answer** or **successful task completion**


In [ ]:
from collections import deque
import heapq
import random

def make_grid(width, height, walls):
    return set(walls)

def in_bounds(x, y, width, height):
    return 0 <= x < width and 0 <= y < height

def neighbors(pos, blocked, width, height):
    x, y = pos
    for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
        nx, ny = x+dx, y+dy
        if in_bounds(nx, ny, width, height) and (nx, ny) not in blocked:
            yield (nx, ny)

def reconstruct_path(parent, start, goal):
    if start == goal:
        return [start]
    if goal not in parent:
        return None
    cur = goal
    path = [cur]
    while cur != start:
        cur = parent[cur]
        path.append(cur)
    path.reverse()
    return path

def render_grid(width, height, blocked, start, goal, path=None):
    path = set(path or [])
    lines = []
    for y in range(height):
        row = []
        for x in range(width):
            p = (x, y)
            if p == start:
                row.append("S")
            elif p == goal:
                row.append("G")
            elif p in blocked:
                row.append("#")
            elif p in path:
                row.append("·")
            else:
                row.append(" ")
        lines.append("".join(row))
    return "\n".join(lines)

# A small maze-like grid
W, H = 16, 9
walls = [
    (2,1),(3,1),(4,1),(5,1),(6,1),
    (6,2),(6,3),(6,4),(6,5),
    (9,0),(9,1),(9,2),(9,3),(9,4),(9,5),
    (12,3),(13,3),(14,3),
    (2,6),(3,6),(4,6),(5,6),(6,6),(7,6),(8,6),
    (12,6),(12,7),(12,8),
]
blocked = make_grid(W, H, walls)
start = (0, 0)
goal  = (15, 8)

print(render_grid(W, H, blocked, start, goal))


---
## 2) BFS (Breadth-First Search)

**Idea:** explore all nodes at depth 0, then depth 1, then depth 2, …  
**Guarantee (unweighted graphs):** shortest path (fewest steps).  
**Cost:** can expand many nodes.

We’ll track:
- `expanded`: how many states we expanded
- `frontier_max`: peak frontier size (memory proxy)


In [ ]:
def bfs(start, goal, blocked, width, height):
    q = deque([start])
    parent = {start: None}
    expanded = 0
    frontier_max = 1

    while q:
        frontier_max = max(frontier_max, len(q))
        cur = q.popleft()
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        for nxt in neighbors(cur, blocked, width, height):
            if nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)

    return None, expanded, frontier_max

path_bfs, expanded_bfs, fmax_bfs = bfs(start, goal, blocked, W, H)
print("BFS path length:", None if path_bfs is None else len(path_bfs)-1)
print("BFS states expanded:", expanded_bfs)
print("BFS peak frontier size:", fmax_bfs)
print()
print(render_grid(W, H, blocked, start, goal, path_bfs))


---
## 3) DFS (Depth-First Search)

**Idea:** go deep before backtracking.  
**Pros:** very low memory, can find *a* solution quickly.  
**Cons:** not optimal, can waste time on deep dead-ends.

We’ll implement iterative DFS with a stack.


In [ ]:
def dfs(start, goal, blocked, width, height):
    stack = [start]
    parent = {start: None}
    expanded = 0
    frontier_max = 1

    while stack:
        frontier_max = max(frontier_max, len(stack))
        cur = stack.pop()
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        nxts = list(neighbors(cur, blocked, width, height))
        for nxt in reversed(nxts):  # deterministic-ish behavior
            if nxt not in parent:
                parent[nxt] = cur
                stack.append(nxt)

    return None, expanded, frontier_max

path_dfs, expanded_dfs, fmax_dfs = dfs(start, goal, blocked, W, H)
print("DFS path length:", None if path_dfs is None else len(path_dfs)-1)
print("DFS states expanded:", expanded_dfs)
print("DFS peak frontier size:", fmax_dfs)
print()
print(render_grid(W, H, blocked, start, goal, path_dfs))


---
## 4) A\* Search

A\* expands nodes in increasing order of:

\[
f(n) = g(n) + h(n)
\]

- `g(n)`: cost so far (here: steps taken)
- `h(n)`: heuristic estimate to goal

We’ll use **Manhattan distance**, which is **admissible** for this grid, so A\* is optimal.


In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(start, goal, blocked, width, height, h_fn=manhattan):
    pq = []
    heapq.heappush(pq, (h_fn(start, goal), 0, start))  # (f, g, state)

    parent = {start: None}
    gbest = {start: 0}

    expanded = 0
    frontier_max = 1

    while pq:
        frontier_max = max(frontier_max, len(pq))
        f, g, cur = heapq.heappop(pq)
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        for nxt in neighbors(cur, blocked, width, height):
            cand_g = g + 1
            if nxt not in gbest or cand_g < gbest[nxt]:
                gbest[nxt] = cand_g
                parent[nxt] = cur
                cand_f = cand_g + h_fn(nxt, goal)
                heapq.heappush(pq, (cand_f, cand_g, nxt))

    return None, expanded, frontier_max

path_astar, expanded_astar, fmax_astar = astar(start, goal, blocked, W, H)
print("A* path length:", None if path_astar is None else len(path_astar)-1)
print("A* states expanded:", expanded_astar)
print("A* peak frontier size:", fmax_astar)
print()
print(render_grid(W, H, blocked, start, goal, path_astar))


---
## 5) Quick comparison (BFS vs DFS vs A\*)

- **BFS**: optimal path length, potentially lots of expansion.
- **DFS**: can be much longer / unstable depending on ordering.
- **A\***: optimal path length (here) and usually fewer expansions due to heuristic guidance.


In [ ]:
def summarize(name, path, expanded, fmax):
    length = None if path is None else len(path) - 1
    return {"algo": name, "path_len": length, "expanded": expanded, "peak_frontier": fmax}

summary = [
    summarize("BFS", path_bfs, expanded_bfs, fmax_bfs),
    summarize("DFS", path_dfs, expanded_dfs, fmax_dfs),
    summarize("A*",  path_astar, expanded_astar, fmax_astar),
]
summary


---
## 6) Heuristics as “guidance” (LLM analogy)

In A\*, `h(n)` is the guidance signal.

In many modern agent systems, the LLM provides a similar guidance signal:
- proposes next step / subgoal / tool call,
- ranks options,
- estimates “distance to completion”.

But model guidance is imperfect, so search still matters for:
- backtracking from wrong steps,
- exploring alternatives,
- enforcing constraints (schemas, safety rules, type checks),
- selecting the best candidate via explicit scoring.

Let’s compare A\* with a good heuristic vs a noisy heuristic (imperfect guide).


In [ ]:
def noisy_manhattan(a, b, noise_scale=3.0, seed=42):
    rnd = random.Random(hash((a, b, seed)) & 0xffffffff)
    return manhattan(a, b) + rnd.uniform(-noise_scale, noise_scale)

def h_noisy(a, b):
    return noisy_manhattan(a, b, noise_scale=3.0, seed=42)

_, ex_good, _ = astar(start, goal, blocked, W, H, h_fn=manhattan)
_, ex_noisy, _ = astar(start, goal, blocked, W, H, h_fn=h_noisy)

print("A* (good h)  expanded:", ex_good)
print("A* (noisy h) expanded:", ex_noisy)


---
## 7) Beam Search (bounded best-first; very relevant to LLMs)

Beam search is a **bounded** search strategy:

- Keep only the **top B** partial candidates (“the beam”) at each depth/step.
- Expand each candidate by one action.
- Score the new candidates.
- Keep the best B and repeat.

This is extremely close to how LLM decoding is often described:
- **Beam width B** = how many candidate continuations you keep.

### Key differences vs A\*
- Beam search is not guaranteed to be optimal or complete.
- It can “beam collapse” (beams become similar), or prune away the only path that leads to a solution.
- But it is **simple, parallelizable**, and works well with good scoring.

### For agents
Beam search is a great mental model for:
- “Try multiple tool-call plans in parallel”
- “Keep a few best plans, prune the rest”
- “Iterate until a plan reaches a verified goal”


In [ ]:
def beam_search_grid(start, goal, blocked, width, height, beam_width=5, max_steps=80):
    # Each beam item: (score, pos, path)
    # We'll *minimize* score = g + h + cycle_penalty
    def score(path, pos):
        g = len(path) - 1
        h = manhattan(pos, goal)
        # penalize revisits a bit (discourages loops)
        cycle_pen = 0.25 * (len(path) - len(set(path)))
        return g + h + cycle_pen

    beam = [(score([start], start), start, [start])]
    expanded = 0

    for step in range(max_steps):
        # If any beam item hits goal, return the best one among goal-reached
        goals = [(s, pth) for (s, pos, pth) in beam if pos == goal]
        if goals:
            goals.sort(key=lambda t: t[0])
            return goals[0][1], expanded

        candidates = []
        for s, pos, path in beam:
            expanded += 1
            for nxt in neighbors(pos, blocked, width, height):
                new_path = path + [nxt]
                candidates.append((score(new_path, nxt), nxt, new_path))

        # Keep top B candidates
        candidates.sort(key=lambda t: t[0])
        beam = candidates[:beam_width]

        if not beam:
            break

    return None, expanded

# Try a few beam widths
for B in [1, 2, 5, 10, 30]:
    path_beam, ex_beam = beam_search_grid(start, goal, blocked, W, H, beam_width=B, max_steps=200)
    print(f"Beam width={B:>2}  found={path_beam is not None}  "
          f"path_len={None if path_beam is None else len(path_beam)-1}  expanded={ex_beam}")


### Visualize one beam-search solution

Try beam width 1 (greedy-ish), then a larger width. You should see that larger beams
are more likely to find a good solution, at the cost of more expansion.


In [ ]:
B = 5
path_beam, ex_beam = beam_search_grid(start, goal, blocked, W, H, beam_width=B, max_steps=200)
print("Beam width:", B, "expanded:", ex_beam, "path_len:", None if path_beam is None else len(path_beam)-1)
print()
print(render_grid(W, H, blocked, start, goal, path_beam))


---
## 8) Why search is important in the LLM era (practical)

**Planner–Executor–Tool systems** often look like:
1. Expand candidate plans / next actions (LLM proposes)
2. Score / verify / simulate (tools + constraints)
3. Select best and continue (search control)

Where algorithms map:
- **DFS**: “commit to one plan until it fails, then backtrack”
- **BFS**: “try all short plans first” (great when the space is small)
- **A\***: “best-first” with explicit cost + heuristic guidance
- **Beam search**: “keep a few best partial plans” (very practical for LLM agents)

Search provides:
- robustness (recovery, backtracking),
- control (limits on branching/cost/risk),
- correctness hooks (constraints, verifiers),
- and better outcomes than a single-shot guess.


---
## 9) Recommended libraries / solvers

**Graph search / shortest path**
- `networkx` — excellent graph algorithms (BFS/DFS/shortest paths)
- `heapq` — what we used; often enough for custom A\* in production

**Planning (PDDL-style)**
- `pyperplan` — lightweight classical planner
- `unified-planning` — broader planning interface with multiple backends

**Constraint optimization (often paired with planning)**
- Google `ortools` — CP-SAT solver, routing, scheduling

Rule of thumb:
- For understanding: implement BFS/DFS/A\*/beam once.
- For real systems: use libraries and invest in **state modeling + scoring + constraints**.
